# 07 · Fine-Tuning, PEFT & LoRA

> **Source notes:** `FineTuning.md`

When is fine-tuning the right call? This notebook covers:
- **Decision tree** — should you fine-tune at all?
- **LoRA math visualised** — low-rank decomposition and parameter savings
- **Dataset preparation** — building and validating a fine-tuning dataset
- **PEFT config** — ready-to-adapt LoRA training setup
- **Distillation workflow** — generating synthetic data from a larger model

> GPU recommended for actual training. Math and dataset cells run on CPU.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','numpy','matplotlib','transformers','peft','datasets','ollama','-q'],check=True)
print('Packages ready.')
import numpy as np, matplotlib.pyplot as plt, json, re

## 1 · The Fine-Tuning Decision Tree

```
Failing output format?       -> Prompt engineering / structured output
Missing domain facts?         -> RAG (fine-tuning memorises facts poorly)
Style/behaviour failure?      -> Fine-tuning v
Too slow/expensive?           -> Distillation v
Proprietary syntax/DSL?       -> Fine-tuning v
Reasoning failures?           -> Better model or CoT
```

In [ ]:
def fine_tune_decision(symptoms):
    if symptoms.get('format_failure'):    return 'PROMPT ENGINEERING -- use structured output mode.'
    if symptoms.get('fact_gap'):          return 'RAG -- facts go stale in model weights.'
    if symptoms.get('reasoning_failure'): return 'BETTER MODEL or COT -- fine-tuning does not fix reasoning.'
    if symptoms.get('latency_cost'):      return 'DISTILLATION -- fine-tune smaller model on teacher outputs.'
    if symptoms.get('specialised_syntax'): return 'FINE-TUNING v -- DSL not in pretraining data.'
    if symptoms.get('style_failure'):     return 'FINE-TUNING v -- style is weight-level; prompts only partially lock it in.'
    return 'NEED MORE INFO -- describe the failure mode.'

scenarios = [
    ({'format_failure': True},    'Model uses prose instead of JSON'),
    ({'fact_gap': True},          'Model missing internal product catalogue'),
    ({'style_failure': True},     'Cannot maintain brand voice via prompting'),
    ({'latency_cost': True},      'GPT-4 is perfect but too expensive at scale'),
    ({'specialised_syntax': True},'Model cannot generate proprietary config format'),
]
for symptoms, desc in scenarios:
    print(f'Problem:  {desc}'); print(f'Decision: {fine_tune_decision(symptoms)}'); print()

## 2 · LoRA Math — Low-Rank Decomposition

LoRA adds: **W' = W + BA** where A in R(r x k), B in R(d x r), r << min(d,k).
Trains r(d+k) params instead of d*k — typically 0.1-3% of full params.

In [ ]:
def lora_stats(d, k, r):
    full = d * k; lora = r * (d + k)
    return {'full': full, 'lora': lora, 'pct': lora/full*100, 'red': full/lora}

configs = [('7B Q-proj 4096x4096', 4096, 4096), ('7B FFN-up 4096x11008', 4096, 11008), ('70B Q-proj 8192x8192', 8192, 8192)]
print(f'{"Layer":<28} {"r":>4} {"Full params":>14} {"LoRA params":>13} {"Pct":>7} {"Reduction":>10}')
print('-'*82)
for name, d, k in configs:
    for r in [8, 64]:
        s = lora_stats(d, k, r)
        print(f'{name:<28} {r:>4} {s["full"]:>14,} {s["lora"]:>13,} {s["pct"]:>7.2f}% {s["red"]:>9.0f}x')
    print()

d, k = 4096, 4096
ranks = range(1, 128)
pcts  = [lora_stats(d, k, r)["pct"] for r in ranks]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(ranks), pcts, lw=2)
for rm in [4, 8, 16, 32, 64]:
    p = lora_stats(d, k, rm)['pct']
    ax.scatter([rm], [p], s=50, zorder=5)
    ax.annotate(f'r={rm}\n{p:.1f}%', (rm, p), xytext=(4,4), textcoords='offset points', fontsize=8)
ax.set_xlabel('LoRA rank r'); ax.set_ylabel('Trainable params %')
ax.set_title('LoRA Trainable Parameters vs Rank (7B Q-proj 4096x4096)')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('lora_rank.png', dpi=120); plt.show()

## 3 · Dataset Preparation & Validation

In [ ]:
TRAINING_DATA = [
    {'system': 'Parse orders to JSON: {items:[{name,size,qty,modifiers}],delivery_address,note}',
     'user':   'Two large Margheritas, extra cheese, to 42 Maple Street.',
     'assistant': json.dumps({'items':[{'name':'Margherita','size':'large','qty':2,'modifiers':['extra cheese']}],'delivery_address':'42 Maple Street','note':None})},
    {'system': 'Parse orders to JSON: {items:[{name,size,qty,modifiers}],delivery_address,note}',
     'user':   'Small Pepperoni for collection, gluten-free base please.',
     'assistant': json.dumps({'items':[{'name':'Pepperoni','size':'small','qty':1,'modifiers':['gluten-free base']}],'delivery_address':None,'note':'collection'})},
    {'system': 'Parse orders to JSON: {items:[{name,size,qty,modifiers}],delivery_address,note}',
     'user':   'What time do you close?',
     'assistant': json.dumps({'error':'not_an_order','message':'This does not appear to be a pizza order.'})},
]
print('Dataset validation:')
errors = 0
for i, row in enumerate(TRAINING_DATA):
    try:
        json.loads(row['assistant']); print(f'  [{i+1}] OK  {row["user"][:55]}')
    except json.JSONDecodeError as e:
        print(f'  [{i+1}] ERR {e}'); errors += 1
print(f'\n{len(TRAINING_DATA)-errors}/{len(TRAINING_DATA)} valid.')
with open('ft_dataset.jsonl','w') as f:
    [f.write(json.dumps(r)+'\n') for r in TRAINING_DATA]
print('Exported ft_dataset.jsonl')

## 4 · PEFT / LoRA Config

Key hyperparameters:
- `r=16` — rank (default; lower=fewer params, higher=more capacity)
- `lora_alpha=32` — effective scale = alpha/r = 2.0
- `target_modules` — which matrices to adapt (Q and V projections for attention)
- `lora_dropout=0.05` — regularisation

Actual training requires GPU with ~16 GB VRAM for a 7B model.

In [ ]:
from peft import LoraConfig, TaskType
from transformers import TrainingArguments

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM, inference_mode=False,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','v_proj'], bias='none',
)
train_args = TrainingArguments(
    output_dir='./ft-lora-out', num_train_epochs=3,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, fp16=True, logging_steps=10,
    save_strategy='epoch', warmup_ratio=0.05, lr_scheduler_type='cosine', report_to='none',
)
d = 4096
trainable = lora_cfg.r * (d + d) * 2  # Q+V for one layer
full = d * d * 2
print('LoRA Config:')
print(f'  r={lora_cfg.r}  alpha={lora_cfg.lora_alpha}  scale={lora_cfg.lora_alpha/lora_cfg.r:.1f}')
print(f'  Trainable per layer: {trainable:,} ({trainable/full:.2%} of full)')
print('\nTo train:')
print('  model = AutoModelForCausalLM.from_pretrained(...)')
print('  model = get_peft_model(model, lora_cfg)')
print('  # then use SFTTrainer from trl or standard Trainer')

## 5 · Distillation — Synthetic Training Data from Teacher

In [ ]:
import ollama
MODEL = 'phi3:mini'
TEACHER_SYS = 'Generate a realistic pizza order variant and its correct JSON parse. Respond with valid JSON only: {"input": "<order text>", "output": {"items": [...], "delivery_address": ..., "note": ...}}'
SEEDS = [
    'A large Margherita and two Garlic Breads to 5 Elm Road.',
    'One medium Pepperoni, no onions, for collection.',
    'Three small Veggie Supremes to Flat 4, 12 Oak St -- leave at door.',
]
dataset = []
print('Generating synthetic training data from teacher model...\n')
for seed in SEEDS:
    resp = ollama.chat(model=MODEL, messages=[{'role':'system','content':TEACHER_SYS},{'role':'user','content':f'Seed: {seed}'}], options={'temperature':0.8,'num_predict':200})
    raw = resp['message']['content'].strip()
    m   = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try:
            ex = json.loads(m.group()); dataset.append(ex)
            print(f'  OK  {str(ex.get("input",""))[:65]}')
        except: print(f'  FAIL parse -- seed: {seed[:50]}')
    else: print(f'  FAIL no JSON -- seed: {seed[:50]}')
print(f'\nGenerated {len(dataset)} examples. Scale to 500-5000 for a real distillation run.')

## Summary

| Concept | Key Takeaway |
|---|---|
| Decision tree | Most problems -> prompting or RAG first; fine-tune last |
| LoRA math | r(d+k) params vs d*k full -- ~1% at r=16 |
| Data > rank | Data quality beats hyperparameter tuning |
| Distillation | Teacher-generated data is most scalable path |
| Eval gate | Run evaluation metrics before/after to measure actual gain |

**Next:** [SafetyAndHallucination/notebook.ipynb](../SafetyAndHallucination/notebook.ipynb)